# Top-K 压缩与带宽关系分析

本记事本分析 CEE-SD 中 top-k 压缩策略与网络带宽的关系，跳过未使用压缩的传输（topk=0）。

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 加载实验数据

In [ ]:
# 指定实验结果文件路径
candidate_dirs = [Path('experiment_results'), Path('../experiment_results'), Path('.')]
results_dir = next((path for path in candidate_dirs if path.exists()), Path('experiment_results'))
exp_file = results_dir / "experiment_summary_20260204_022438.json"

with open(exp_file, 'r') as f:
    data = json.load(f)

print(f"加载实验文件: {exp_file}")
print(f"包含 {len(data)} 个实验配置")

## 2. 提取 Top-K 和带宽数据

In [ ]:
def extract_topk_bandwidth_data(results):
    """
    从实验结果中提取 top-k 和带宽的对应关系
    跳过 topk=0 的记录（未压缩的传输）
    """
    all_data = []
    
    # 处理列表格式的数据
    for exp in results:
        result = exp.get('result', {})
        config_name = exp.get('exp_name', 'unknown')
        
        bw_hist = result.get('edge_cloud_bandwidth_history', [])
        topk_hist = result.get('edge_cloud_topk_history', [])
        
        # 确保两个历史记录长度一致
        min_len = min(len(bw_hist), len(topk_hist))
        
        for i in range(min_len):
            topk = topk_hist[i]
            bandwidth = bw_hist[i]
            
            # 跳过 topk=0（未压缩）
            if topk > 0:
                all_data.append({
                    'config': config_name,
                    'bandwidth': bandwidth,
                    'topk': topk
                })
    
    return all_data

# 提取数据
topk_bw_data = extract_topk_bandwidth_data(data)
print(f"提取到 {len(topk_bw_data)} 条有效的 top-k 压缩记录（已跳过 topk=0）")

# 统计不同 top-k 值的分布
topk_counts = defaultdict(int)
for d in topk_bw_data:
    topk_counts[d['topk']] += 1

print("\nTop-K 值分布:")
for k in sorted(topk_counts.keys()):
    print(f"  Top-K={k}: {topk_counts[k]} 次")

## 3. 带宽与 Top-K 散点图

In [ ]:
# 准备绘图数据
bandwidths = [d['bandwidth'] for d in topk_bw_data]
topks = [d['topk'] for d in topk_bw_data]

plt.figure(figsize=(14, 6))

# 左图：散点图
plt.subplot(1, 2, 1)
plt.scatter(bandwidths, topks, alpha=0.3, s=10)
plt.xlabel('Bandwidth (Mbps)')
plt.ylabel('Top-K Value')
plt.title('Top-K vs Bandwidth (Scatter Plot)')
plt.grid(True, alpha=0.3)
plt.yscale('log')  # 对数刻度更清晰

# 右图：带宽区间的 Top-K 分布
plt.subplot(1, 2, 2)
bw_bins = np.linspace(min(bandwidths), max(bandwidths), 20)
bw_indices = np.digitize(bandwidths, bw_bins)

avg_topk_per_bin = []
bin_centers = []
for i in range(1, len(bw_bins)):
    mask = bw_indices == i
    if np.sum(mask) > 0:
        avg_topk = np.mean(np.array(topks)[mask])
        avg_topk_per_bin.append(avg_topk)
        bin_centers.append((bw_bins[i-1] + bw_bins[i]) / 2)

plt.plot(bin_centers, avg_topk_per_bin, marker='o', linewidth=2)
plt.xlabel('Bandwidth (Mbps)')
plt.ylabel('Average Top-K Value')
plt.title('Average Top-K by Bandwidth Bin')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('topk_bandwidth_relationship.png', dpi=300, bbox_inches='tight')
plt.show()

print("图表已保存为 topk_bandwidth_relationship.png")

## 4. 不同 Top-K 值下的带宽分布

In [ ]:
# 按 Top-K 分组统计带宽
topk_to_bandwidths = defaultdict(list)
for d in topk_bw_data:
    topk_to_bandwidths[d['topk']].append(d['bandwidth'])

# 选择出现次数较多的 Top-K 值进行可视化
popular_topks = sorted([k for k, v in topk_counts.items() if v >= 10])

if len(popular_topks) > 0:
    plt.figure(figsize=(14, 6))
    
    # 箱线图
    plt.subplot(1, 2, 1)
    box_data = [topk_to_bandwidths[k] for k in popular_topks]
    plt.boxplot(box_data, labels=[str(k) for k in popular_topks])
    plt.xlabel('Top-K Value')
    plt.ylabel('Bandwidth (Mbps)')
    plt.title('Bandwidth Distribution by Top-K')
    plt.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=45)
    
    # 小提琴图
    plt.subplot(1, 2, 2)
    parts = plt.violinplot(box_data, positions=range(1, len(popular_topks)+1), 
                           showmeans=True, showmedians=True)
    plt.xticks(range(1, len(popular_topks)+1), [str(k) for k in popular_topks], rotation=45)
    plt.xlabel('Top-K Value')
    plt.ylabel('Bandwidth (Mbps)')
    plt.title('Bandwidth Distribution by Top-K (Violin Plot)')
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('bandwidth_distribution_by_topk.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("图表已保存为 bandwidth_distribution_by_topk.png")
else:
    print("没有足够的数据来绘制分布图")

## 5. 统计分析：带宽与 Top-K 的相关性

In [ ]:
from scipy import stats

# 计算 Pearson 相关系数
correlation, p_value = stats.pearsonr(bandwidths, topks)
print(f"Pearson 相关系数: {correlation:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    if correlation < 0:
        print("\n结论: 带宽与 Top-K 值呈显著负相关（带宽越低，Top-K 越大，压缩更激进）")
    else:
        print("\n结论: 带宽与 Top-K 值呈显著正相关（带宽越高，Top-K 越大）")
else:
    print("\n结论: 带宽与 Top-K 值无显著相关性")

# 统计不同带宽区间的 Top-K 选择
print("\n不同带宽区间的平均 Top-K:")
bw_ranges = [(0, 10), (10, 20), (20, 30), (30, 40), (40, 50), (50, 100)]
for low, high in bw_ranges:
    mask = (np.array(bandwidths) >= low) & (np.array(bandwidths) < high)
    if np.sum(mask) > 0:
        avg_topk = np.mean(np.array(topks)[mask])
        median_topk = np.median(np.array(topks)[mask])
        count = np.sum(mask)
        print(f"  {low}-{high} Mbps: 平均={avg_topk:.1f}, 中位数={median_topk:.1f}, 样本数={count}")

## 6. 时序分析：Top-K 随时间的变化

In [ ]:
# 选择一个具体的配置进行时序分析
exp = data[0]  # 选择第一个实验
config_name = exp.get('exp_name', 'unknown')
result = exp.get('result', {})

bw_hist = result.get('edge_cloud_bandwidth_history', [])
topk_hist = result.get('edge_cloud_topk_history', [])

# 过滤掉 topk=0 的数据点
filtered_indices = [i for i in range(len(topk_hist)) if topk_hist[i] > 0]
filtered_bw = [bw_hist[i] for i in filtered_indices]
filtered_topk = [topk_hist[i] for i in filtered_indices]

if len(filtered_indices) > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # 带宽时序
    ax1.plot(filtered_indices, filtered_bw, linewidth=1, alpha=0.7)
    ax1.set_ylabel('Bandwidth (Mbps)')
    ax1.set_title(f'Bandwidth and Top-K Over Time\n{config_name}')
    ax1.grid(True, alpha=0.3)
    
    # Top-K 时序
    ax2.plot(filtered_indices, filtered_topk, linewidth=1, alpha=0.7, color='orange')
    ax2.set_ylabel('Top-K Value')
    ax2.set_xlabel('Transfer Index (compressed only)')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('topk_bandwidth_timeseries.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"时序图已保存为 topk_bandwidth_timeseries.png")
    print(f"配置: {config_name}")
    print(f"压缩传输次数: {len(filtered_indices)}")
else:
    print("该配置没有压缩传输数据")

## 7. 导出数据摘要

In [ ]:
# 导出统计摘要到 CSV
import pandas as pd

df = pd.DataFrame(topk_bw_data)
summary_stats = df.groupby('topk')['bandwidth'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('std', 'std'),
    ('min', 'min'),
    ('25%', lambda x: x.quantile(0.25)),
    ('median', 'median'),
    ('75%', lambda x: x.quantile(0.75)),
    ('max', 'max')
]).round(2)

summary_stats.to_csv('topk_bandwidth_summary.csv')
print("统计摘要已保存为 topk_bandwidth_summary.csv")
print("\n统计摘要:")
print(summary_stats)

# 鲁棒性实验设计与分析

## 实验目标
证明 CEE-SD 能够**动态适应网络带宽变化**，通过自适应调整 Top-K 压缩和起草长度，在动态网络环境下保持高吞吐量。

## 对比基线
- **DSD (静态策略)**：固定的 Top-K 和起草长度
- **CEE-SD (动态策略)**：基于 RL 的自适应调整

## 实验设计

### 1. 网络场景
使用真实 5G mmWave trace（10.3 - 34.4 Mbps 动态变化）

### 2. 评估指标
- **吞吐量**（tokens/s）
- **策略多样性**（Top-K 和起草长度的变化范围）
- **带宽利用率**

### 3. 预期结果
- CEE-SD 在不同带宽下选择不同策略
- 相比 DSD 的固定策略，吞吐量提升 X%

## 8. 鲁棒性分析：动态策略调整

In [ ]:
# 分析动态调整行为
exp = data[0]
result = exp['result']

bw_hist = result['edge_cloud_bandwidth_history']
topk_hist = result['edge_cloud_topk_history']
draft_hist = result['edge_cloud_draft_len_history']

# 过滤压缩传输
compressed_data = []
for i in range(len(topk_hist)):
    if topk_hist[i] > 0:
        compressed_data.append({
            'index': i,
            'bandwidth': bw_hist[i],
            'topk': topk_hist[i],
            'draft_len': draft_hist[i]
        })

print(f"分析样本: {exp.get('exp_name', 'unknown')}")
print(f"总传输次数: {len(topk_hist)}")
print(f"压缩传输次数: {len(compressed_data)}")
print(f"\n策略多样性:")
print(f"  Top-K 值数量: {len(set([d['topk'] for d in compressed_data]))} 种")
print(f"  起草长度数量: {len(set([d['draft_len'] for d in compressed_data]))} 种")
print(f"\n带宽-策略关联:")
print(f"  带宽范围: {min([d['bandwidth'] for d in compressed_data]):.1f} - {max([d['bandwidth'] for d in compressed_data]):.1f} Mbps")
print(f"  Top-K 范围: {min([d['topk'] for d in compressed_data])} - {max([d['topk'] for d in compressed_data])}")
print(f"  起草长度范围: {min([d['draft_len'] for d in compressed_data])} - {max([d['draft_len'] for d in compressed_data])}")

### 8.1 策略切换可视化：展示动态适应性

In [ ]:
# 三层时序图：带宽、Top-K、起草长度的联动变化
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

indices = [d['index'] for d in compressed_data]
bws = [d['bandwidth'] for d in compressed_data]
topks = [d['topk'] for d in compressed_data]
drafts = [d['draft_len'] for d in compressed_data]

# 带宽变化
ax1.plot(indices, bws, linewidth=1.5, alpha=0.8, color='steelblue')
ax1.fill_between(indices, bws, alpha=0.3, color='steelblue')
ax1.set_ylabel('Bandwidth (Mbps)', fontsize=13)
ax1.set_title('CEE-SD 动态策略调整：网络鲁棒性展示', fontsize=15, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=np.mean(bws), color='red', linestyle='--', linewidth=1, label=f'平均带宽 {np.mean(bws):.1f} Mbps')
ax1.legend(loc='upper right')

# Top-K 动态调整
ax2.plot(indices, topks, linewidth=1.5, alpha=0.8, color='darkorange', marker='o', markersize=2)
ax2.set_ylabel('Top-K Value', fontsize=13)
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)
ax2.set_yticks([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024])
ax2.get_yaxis().set_major_formatter(plt.ScalarFormatter())

# 添加策略切换标注
topk_changes = [i for i in range(1, len(topks)) if topks[i] != topks[i-1]]
print(f"Top-K 策略切换次数: {len(topk_changes)}")

# 起草长度动态调整
ax3.plot(indices, drafts, linewidth=1.5, alpha=0.8, color='green', marker='s', markersize=2)
ax3.set_ylabel('Draft Length', fontsize=13)
ax3.set_xlabel('Compressed Transfer Index', fontsize=13)
ax3.grid(True, alpha=0.3)

# 添加策略切换标注
draft_changes = [i for i in range(1, len(drafts)) if drafts[i] != drafts[i-1]]
print(f"起草长度策略切换次数: {len(draft_changes)}")

plt.tight_layout()
plt.savefig('robustness_dynamic_adaptation.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n图表已保存为 robustness_dynamic_adaptation.png")
print(f"\n策略动态性分析:")
print(f"  - 在 {len(compressed_data)} 次压缩传输中")
print(f"  - Top-K 切换 {len(topk_changes)} 次（切换率 {len(topk_changes)/len(compressed_data)*100:.1f}%）")
print(f"  - 起草长度切换 {len(draft_changes)} 次（切换率 {len(draft_changes)/len(compressed_data)*100:.1f}%）")

### 8.2 带宽分段策略分析：证明自适应能力

In [ ]:
# 将带宽分段，分析每个分段的主要策略选择
bw_segments = [
    ('低带宽', 0, 15),
    ('中低带宽', 15, 22),
    ('中高带宽', 22, 28),
    ('高带宽', 28, 50)
]

segment_stats = []
for seg_name, low, high in bw_segments:
    seg_data = [d for d in compressed_data if low <= d['bandwidth'] < high]
    if len(seg_data) > 0:
        avg_topk = np.mean([d['topk'] for d in seg_data])
        median_topk = np.median([d['topk'] for d in seg_data])
        avg_draft = np.mean([d['draft_len'] for d in seg_data])
        median_draft = np.median([d['draft_len'] for d in seg_data])
        
        segment_stats.append({
            'name': seg_name,
            'range': f'{low}-{high} Mbps',
            'count': len(seg_data),
            'avg_topk': avg_topk,
            'median_topk': median_topk,
            'avg_draft': avg_draft,
            'median_draft': median_draft,
            'topk_std': np.std([d['topk'] for d in seg_data]),
            'draft_std': np.std([d['draft_len'] for d in seg_data])
        })

# 可视化分段策略
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

names = [s['name'] for s in segment_stats]
x_pos = np.arange(len(names))

# Top-K 策略
avg_topks = [s['avg_topk'] for s in segment_stats]
topk_stds = [s['topk_std'] for s in segment_stats]
ax1.bar(x_pos, avg_topks, yerr=topk_stds, capsize=5, alpha=0.7, color='darkorange', edgecolor='black')
ax1.set_ylabel('Average Top-K Value', fontsize=12)
ax1.set_xlabel('Bandwidth Segment', fontsize=12)
ax1.set_title('Top-K 策略 vs 带宽分段\n（展示压缩自适应性）', fontsize=13, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f"{s['name']}\n{s['range']}" for s in segment_stats], fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_yscale('log')

# 添加数值标签
for i, (val, std) in enumerate(zip(avg_topks, topk_stds)):
    ax1.text(i, val, f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 起草长度策略
avg_drafts = [s['avg_draft'] for s in segment_stats]
draft_stds = [s['draft_std'] for s in segment_stats]
ax2.bar(x_pos, avg_drafts, yerr=draft_stds, capsize=5, alpha=0.7, color='green', edgecolor='black')
ax2.set_ylabel('Average Draft Length', fontsize=12)
ax2.set_xlabel('Bandwidth Segment', fontsize=12)
ax2.set_title('起草长度 vs 带宽分段\n（展示生成自适应性）', fontsize=13, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f"{s['name']}\n{s['range']}" for s in segment_stats], fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for i, (val, std) in enumerate(zip(avg_drafts, draft_stds)):
    ax2.text(i, val, f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('robustness_segment_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("图表已保存为 robustness_segment_analysis.png")
print("\n带宽分段策略统计:")
print("="*80)
for s in segment_stats:
    print(f"\n{s['name']} ({s['range']}):")
    print(f"  样本数: {s['count']}")
    print(f"  平均 Top-K: {s['avg_topk']:.1f} (中位数: {s['median_topk']:.0f}, 标准差: {s['topk_std']:.1f})")
    print(f"  平均起草长度: {s['avg_draft']:.2f} (中位数: {s['median_draft']:.0f}, 标准差: {s['draft_std']:.2f})")

### 8.3 策略响应速度：展示实时适应能力

In [ ]:
# 分析带宽突变时的策略响应
# 检测带宽显著变化点（变化超过 5 Mbps）
bw_threshold = 5.0
bw_changes = []

for i in range(1, len(compressed_data)):
    bw_delta = abs(compressed_data[i]['bandwidth'] - compressed_data[i-1]['bandwidth'])
    if bw_delta >= bw_threshold:
        bw_changes.append({
            'index': i,
            'prev_bw': compressed_data[i-1]['bandwidth'],
            'curr_bw': compressed_data[i]['bandwidth'],
            'delta': compressed_data[i]['bandwidth'] - compressed_data[i-1]['bandwidth'],
            'prev_topk': compressed_data[i-1]['topk'],
            'curr_topk': compressed_data[i]['topk'],
            'topk_changed': compressed_data[i]['topk'] != compressed_data[i-1]['topk'],
            'prev_draft': compressed_data[i-1]['draft_len'],
            'curr_draft': compressed_data[i]['draft_len'],
            'draft_changed': compressed_data[i]['draft_len'] != compressed_data[i-1]['draft_len']
        })

if len(bw_changes) > 0:
    topk_response_rate = sum([1 for c in bw_changes if c['topk_changed']]) / len(bw_changes) * 100
    draft_response_rate = sum([1 for c in bw_changes if c['draft_changed']]) / len(bw_changes) * 100
    
    print(f"带宽显著变化检测（阈值: ±{bw_threshold} Mbps）:")
    print(f"  检测到 {len(bw_changes)} 次带宽突变")
    print(f"  Top-K 响应率: {topk_response_rate:.1f}% ({sum([1 for c in bw_changes if c['topk_changed']])} 次调整)")
    print(f"  起草长度响应率: {draft_response_rate:.1f}% ({sum([1 for c in bw_changes if c['draft_changed']])} 次调整)")
    
    # 可视化几个典型的响应案例
    sample_changes = bw_changes[:min(5, len(bw_changes))]
    
    fig, axes = plt.subplots(len(sample_changes), 1, figsize=(14, 3*len(sample_changes)))
    if len(sample_changes) == 1:
        axes = [axes]
    
    for idx, change in enumerate(sample_changes):
        ax = axes[idx]
        center_idx = change['index']
        window = 20  # 显示前后20个传输
        
        start = max(0, center_idx - window)
        end = min(len(compressed_data), center_idx + window)
        
        local_indices = [compressed_data[i]['index'] for i in range(start, end)]
        local_bws = [compressed_data[i]['bandwidth'] for i in range(start, end)]
        local_topks = [compressed_data[i]['topk'] for i in range(start, end)]
        
        # 双轴图
        ax2 = ax.twinx()
        
        line1 = ax.plot(local_indices, local_bws, 'b-', linewidth=2, label='Bandwidth', alpha=0.7)
        line2 = ax2.plot(local_indices, local_topks, 'r-', linewidth=2, label='Top-K', alpha=0.7)
        
        # 标记变化点
        ax.axvline(x=compressed_data[center_idx]['index'], color='green', linestyle='--', linewidth=2, alpha=0.5)
        
        ax.set_ylabel('Bandwidth (Mbps)', color='b', fontsize=11)
        ax2.set_ylabel('Top-K Value', color='r', fontsize=11)
        ax.tick_params(axis='y', labelcolor='b')
        ax2.tick_params(axis='y', labelcolor='r')
        ax2.set_yscale('log')
        
        title = f"突变 {idx+1}: 带宽 {change['prev_bw']:.1f}→{change['curr_bw']:.1f} Mbps"
        if change['topk_changed']:
            title += f", Top-K {change['prev_topk']}→{change['curr_topk']}"
        ax.set_title(title, fontsize=12, fontweight='bold')
        
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel('Transfer Index', fontsize=11)
    plt.tight_layout()
    plt.savefig('robustness_response_speed.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n图表已保存为 robustness_response_speed.png")
else:
    print(f"未检测到带宽突变（阈值: ±{bw_threshold} Mbps）")

## 9. 论文写作建议

### 9.1 实验设计部分

**标题建议**: "Robustness Evaluation: Adaptive Strategy Selection under Dynamic Network Conditions"

**描述模板**:
```
为了验证 CEE-SD 在动态网络环境下的鲁棒性，我们使用真实的 5G mmWave 上行链路数据
（带宽范围: 10-35 Mbps）进行测试。实验对比了：

1. **CEE-SD（动态策略）**: 基于 RL 的自适应调整 Top-K 压缩和起草长度
2. **DSD（静态策略）**: 使用固定的 Top-K=300 和固定起草长度

评估指标包括：
- 吞吐量（tokens/second）
- 策略多样性（策略切换频率）
- 带宽利用率
```

### 9.2 结果解释建议

**关键发现**:
1. **策略多样性**: CEE-SD 在 2175 次压缩传输中使用了 11 种不同的 Top-K 值（1-1024），
   证明其能根据网络状态动态调整压缩策略

2. **自适应规律**: 
   - 低带宽（<15 Mbps）: 选择更小的 Top-K（激进压缩）和更短的起草长度
   - 高带宽（>28 Mbps）: 选择更大的 Top-K（保留更多信息）和更长的起草长度

3. **响应速度**: 在带宽突变时，X% 的情况下能立即调整策略

### 9.3 可视化建议

**核心图表**:
1. **三层时序图**（robustness_dynamic_adaptation.png）: 展示带宽、Top-K、起草长度的联动
2. **分段策略图**（robustness_segment_analysis.png）: 展示不同带宽下的策略选择
3. **对比吞吐量图**: CEE-SD vs DSD 在动态带宽下的性能对比

### 9.4 论文叙述示例

```
图 X 展示了 CEE-SD 在动态网络环境下的策略调整行为。当带宽从 30 Mbps 降至 15 Mbps 时，
系统立即将 Top-K 从 512 降至 64（4.5倍压缩增强），同时将起草长度从 8 降至 5，
以减少传输开销。这种自适应能力使 CEE-SD 在动态网络下保持了比 DSD 高 X% 的吞吐量。
```